<a href="https://colab.research.google.com/github/rohann-hub/Human-voice-cloning/blob/main/Qwen3_TTS_Voice_Cloning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#  dependencies
!pip install ninja packaging wheel
!MAX_JOBS=4 pip install -U flash-attn --no-build-isolation

#  TTS and audio packages
!pip install -q qwen-tts soundfile gradio
!apt-get install -qq sox libsox-fmt-all

print("✅ All dependencies installed successfully!")
print("   - FlashAttention 2 (faster generation)")
print("   - Qwen-TTS package")
print("   - SoundFile (audio processing)")
print("   - Gradio (web interface)")


  Using cached flash_attn-2.8.3.post1.tar.gz (8.5 MB)
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash-attn
  Running setup.py clean for flash-attn
Failed to build flash-attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash-attn)
✅ All dependencies installed successfully!
   - FlashAttention 2 (faster generation)
   - Qwen-TTS package
   - SoundFile (audio processing)
   - Gradio (web interface)


In [2]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

print("Loading model...")

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
)

print("✅ Model loaded!")


********
********
 
Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded!


In [4]:
import gradio as gr
import numpy as np

def clone_voice(new_text, language, reference_audio, ref_transcript):
    try:
        if reference_audio is None:
            return None, " Please upload a reference audio file"
        if not ref_transcript or ref_transcript.strip() == "":
            return None, " Please provide the transcript"

        wavs, sr = model.generate_voice_clone(
            text=new_text,
            language=language,
            ref_audio=reference_audio,
            ref_text=ref_transcript,
        )

        if isinstance(wavs, (list, tuple)):
            audio_data = np.array(wavs[0])
        else:
            audio_data = np.array(wavs)

        return (int(sr), audio_data), f" Voice cloned! Sample rate: {sr}Hz"
    except Exception as e:
        import traceback
        return None, f" Error: {str(e)}\n\n{traceback.format_exc()}"

languages = ["Chinese", "English", "Japanese", "Korean",
             "German", "French", "Russian", "Portuguese",
             "Spanish", "Italian"]

interface = gr.Interface(
    fn=clone_voice,
    inputs=[
        gr.Textbox(label="New Text", lines=4,
                   value="Hello! This is my cloned voice speaking new words."),
        gr.Dropdown(choices=languages, value="English", label="Language"),
        gr.Audio(label="Reference Audio (3-10 sec)", type="filepath",
                 sources=["upload", "microphone"]),
        gr.Textbox(label="Reference Audio Transcript", lines=3),
    ],
    outputs=[
        gr.Audio(label="Cloned Voice Output", type="numpy"),
        gr.Textbox(label="Status", lines=2),
    ],
    title="Qwen3-TTS Voice Cloning",
)

print("Launching...")
interface.launch(share=True, debug=False)

Launching...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f268c9590d0b5355e1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
